# Gradient Boosting, Explained Simply
Trees built one after another, each fixing the mistakes of the ones before. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A bike-share scheme recorded 2,000 hours: the hour of day, weekend or not, temperature, rain, and how many bikes were rented.

### 2,000 hours of rentals
- `bike_rentals.csv`: 2,000 hours: hour (0 to 23), weekend (1 = yes), temperature in C, raining (1 = yes), rentals

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "gradient-boosting/bike_rentals.csv")
df = pd.read_csv(url)
print(df.shape)
df.head()

### Train and test sets
- `test_size=0.25`: a quarter of the hours kept for testing

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="rentals")
y = df["rentals"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)
print(len(X_train), "training hours,", len(X_test), "test hours")

## 2. Boosting by hand
Start with the average. Then repeat: measure the errors (the **residuals**), train a small tree to predict them, and add a fraction (the **learning rate**) of its prediction.

### Step one: start with the average
- `np.full(..., y_train.mean())`: the first guess: the average rentals, for every hour
- `lr = 0.3`: the learning rate: add 30% of each correction
- `mean_absolute_error`: the average miss, in bikes

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

lr = 0.3
pred_train = np.full(len(y_train), y_train.mean())
pred_test = np.full(len(y_test), y_train.mean())
print(f"start: test error {mean_absolute_error(y_test, pred_test):.1f} bikes")

### Then add trees, one at a time
- `residuals = y_train - pred_train`: what's still wrong: actual minus current prediction
- `DecisionTreeRegressor(max_depth=3)`: a small tree, trained to predict the residuals
- `pred += lr * small.predict(...)`: add 30% of its correction
- `if step in [...]`: print the error now and then

In [ ]:
for step in range(1, 51):
    residuals = y_train - pred_train
    small = DecisionTreeRegressor(max_depth=3).fit(X_train, residuals)
    pred_train += lr * small.predict(X_train)
    pred_test += lr * small.predict(X_test)
    if step in [1, 2, 5, 10, 25, 50]:
        err = mean_absolute_error(y_test, pred_test)
        print(f"after {step:>2} trees: test error {err:.1f} bikes")

## 3. Gradient boosting with scikit-learn
The same idea, polished: many small trees, a learning rate, and a few extra tricks.

### Tree vs forest vs boosting
- `n_estimators=300`: 300 trees, built one after another
- `learning_rate=0.1`: each tree adds 10% of its correction
- `max_depth=4`: each tree is small
- `score`: R squared: 1 is perfect, 0 is no better than the average

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "one tree": DecisionTreeRegressor(max_depth=6, random_state=0),
    "random forest": RandomForestRegressor(300, random_state=0, n_jobs=-1),
    "gradient boosting": GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.1, max_depth=4, random_state=0)}
for name, m in models.items():
    m.fit(X_train, y_train)
    print(f"{name:>17}: test R squared {m.score(X_test, y_test):.3f}")

### Learning rate vs number of trees
- `staged_predict`: the predictions after 1 tree, 2 trees, 3 trees... up to 500
- `1.0`: big steps: quick, but bumpy, and it starts overfitting
- `0.01`: tiny steps: steady, but still improving after 500 trees
- `0.1`: a good balance

In [ ]:
import matplotlib.pyplot as plt

for lr in [1.0, 0.1, 0.01]:
    gb = GradientBoostingRegressor(n_estimators=500, learning_rate=lr,
                                   max_depth=4, random_state=0)
    gb.fit(X_train, y_train)
    errors = [mean_absolute_error(y_test, p)
              for p in gb.staged_predict(X_test)]
    plt.plot(errors, label=f"learning rate {lr}")
plt.ylim(0, 40); plt.xlabel("number of trees"); plt.ylabel("test error (bikes)")
plt.legend(); plt.show()

## 4. The modern version
HistGradientBoostingRegressor is a much faster version, with **early stopping**: it stops adding trees when a held-out part of the training data stops improving.

### Fast boosting with early stopping
- `max_iter=1000`: allowed up to 1,000 trees
- `early_stopping=True`: keeps 10% of the training data aside, and stops when it stops improving
- `n_iter_`: how many trees it actually built

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb = HistGradientBoostingRegressor(max_iter=1000, learning_rate=0.1,
                                    early_stopping=True, random_state=0)
hgb.fit(X_train, y_train)
print("trees used:", hgb.n_iter_)
print("test R squared:", round(hgb.score(X_test, y_test), 3))

## 5. Mini project: plan the bikes
Predict demand for a few situations.

### How many bikes will be needed?
- `cases`: four situations, with the same columns as training
- `hgb.predict`: the boosted model's predictions

In [ ]:
cases = pd.DataFrame([
    {"hour": 8, "weekend": 0, "temperature": 18, "raining": 0},
    {"hour": 8, "weekend": 0, "temperature": 18, "raining": 1},
    {"hour": 14, "weekend": 1, "temperature": 24, "raining": 0},
    {"hour": 3, "weekend": 0, "temperature": 12, "raining": 0}])
cases["predicted"] = hgb.predict(cases).round()
print(cases)

## Try it yourself
1. Predict rentals for a rainy weekday at 8 am, 10 degrees.
2. In the hand-made booster, try a learning rate of 1.0 and of 0.05. How does the error change after 50 rounds?
3. Try max_depth=1 (tree stumps) in GradientBoostingRegressor with 1,000 trees.